# 🔷 Delta Lake com Apache Spark

## Cenário: Superstore — Varejo nos Estados Unidos

**Fonte de dados:** [Superstore Dataset — Kaggle (vivek468)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
**Licença:** Other  
**Amostra utilizada:** 20 clientes e 20 pedidos de uma rede de varejo norte-americana

---

### Modelo ER

```
┌─────────────────────┐       ┌────────────────────────────────────┐
│      clientes       │       │             pedidos                │
│─────────────────────│       │────────────────────────────────────│
│ customer_id (PK)    │──────<│ order_id                           │
│ customer_name       │       │ customer_id (FK)                   │
│ segment             │       │ order_date                         │
│ city                │       │ ship_date                          │
│ state               │       │ ship_mode                          │
│ region              │       │ product_name                       │
└─────────────────────┘       │ category                           │
                              │ sub_category                       │
                              │ sales      DOUBLE                  │
                              │ quantity   INT                     │
                              │ discount   DOUBLE                  │
                              │ profit     DOUBLE                  │
                              └────────────────────────────────────┘
```

### DDL

```sql
CREATE TABLE clientes (
    customer_id   STRING,
    customer_name STRING,
    segment       STRING,
    city          STRING,
    state         STRING,
    region        STRING
);

CREATE TABLE pedidos (
    order_id      STRING,
    customer_id   STRING,
    order_date    STRING,
    ship_date     STRING,
    ship_mode     STRING,
    product_name  STRING,
    category      STRING,
    sub_category  STRING,
    sales         DOUBLE,
    quantity      INT,
    discount      DOUBLE,
    profit        DOUBLE
);
```

## 1. Configuração do Ambiente

In [1]:
import os
import pyspark
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

CWD = os.getcwd()
PROJECT_ROOT = os.path.dirname(CWD) if os.path.basename(CWD) == 'notebooks' else CWD
DATA_RAW   = os.path.join(PROJECT_ROOT, 'data', 'raw')
DELTA_PATH = os.path.join(PROJECT_ROOT, 'data', 'delta')

builder = (
    SparkSession.builder
    .appName('Delta Lake - Superstore')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print(f'PySpark: {pyspark.__version__}')
print(f'CSV fonte : {DATA_RAW}')
print(f'Delta Lake: {DELTA_PATH}')
print('✅ SparkSession com Delta Lake iniciada!')

:: loading settings :: url = jar:file:/home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabrielmaciel/.ivy2/cache
The jars for the packages stored in: /home/gabrielmaciel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4c763cae-1b6c-4046-a934-3575234539b2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 759ms :: artifacts dl 43ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

PySpark: 3.5.1
CSV fonte : /home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/data/raw
Delta Lake: /home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/data/delta
✅ SparkSession com Delta Lake iniciada!


## 2. Leitura da Fonte de Dados — Kaggle Superstore

Os dados vêm do **Superstore Dataset** (Kaggle) — uma rede de varejo norte-americana com vendas de Furniture, Office Supplies e Technology.  
Usamos uma amostra de **20 clientes** e **20 pedidos** com dados completamente legíveis.

In [2]:
df_clientes = spark.read.csv(
    os.path.join(DATA_RAW, 'sample_clientes.csv'),
    header=True, inferSchema=True
)

df_pedidos = spark.read.csv(
    os.path.join(DATA_RAW, 'sample_pedidos.csv'),
    header=True, inferSchema=True
)

print('👤 Clientes (amostra Superstore):')
df_clientes.show(5, truncate=False)

print('🛒 Pedidos — distribuição por ship_mode:')
df_pedidos.groupBy('ship_mode').count().show()

print('🛒 Pedidos (amostra):')
df_pedidos.select('order_id', 'customer_id', 'product_name', 'category', 'sales', 'profit', 'ship_mode').show(5, truncate=True)

👤 Clientes (amostra Superstore):
+-----------+---------------+---------+---------------+--------------+------+
|customer_id|customer_name  |segment  |city           |state         |region|
+-----------+---------------+---------+---------------+--------------+------+
|CG-12520   |Claire Gute    |Consumer |Henderson      |Kentucky      |South |
|DV-13045   |Darrin Van Huff|Corporate|Los Angeles    |California    |West  |
|SO-20335   |Sean O'Donnell |Consumer |Fort Lauderdale|Florida       |South |
|BH-11710   |Brosina Hoffman|Consumer |Los Angeles    |California    |West  |
|AA-10480   |Andrew Allen   |Consumer |Concord        |North Carolina|South |
+-----------+---------------+---------+---------------+--------------+------+
only showing top 5 rows

🛒 Pedidos — distribuição por ship_mode:


+--------------+-----+
|     ship_mode|count|
+--------------+-----+
|  Second Class|   10|
|Standard Class|   10|
+--------------+-----+

🛒 Pedidos (amostra):
+--------------+-----------+--------------------+---------------+------+-------+------------+
|      order_id|customer_id|        product_name|       category| sales| profit|   ship_mode|
+--------------+-----------+--------------------+---------------+------+-------+------------+
|CA-2016-152156|   CG-12520|Bush Somerset Col...|      Furniture|261.96|41.9136|Second Class|
|CA-2016-152156|   CG-12520|Hon Deluxe Fabric...|      Furniture|731.94|219.582|Second Class|
|CA-2016-138688|   DV-13045|Self-Adhesive Add...|Office Supplies| 14.62| 6.8714|Second Class|
|CA-2014-167164|   AG-10270|Fellowes Super St...|Office Supplies|  55.5|   9.99|Second Class|
|CA-2014-143336|   ZD-21925|          Newell 341|Office Supplies|  8.56| 2.4824|Second Class|
+--------------+-----------+--------------------+---------------+------+-------+--------

## 3. INSERT — Gravando dados no Delta Lake

O método `write.format("delta")` é o equivalente ao `INSERT` no Delta Lake.  
Ele converte o DataFrame em arquivos Parquet e registra a operação no `_delta_log/` como versão 0 (`WRITE`).

In [3]:
df_clientes.write.format('delta').mode('overwrite').save(f'{DELTA_PATH}/clientes')
print('✅ Tabela clientes gravada no Delta Lake')

df_pedidos.write.format('delta').mode('overwrite').save(f'{DELTA_PATH}/pedidos')
print('✅ Tabela pedidos gravada no Delta Lake')

print(f'\nTotal registros inseridos: {spark.read.format("delta").load(f"{DELTA_PATH}/pedidos").count()}')
spark.read.format('delta').load(f'{DELTA_PATH}/pedidos') \
    .select('order_id', 'customer_id', 'product_name', 'category', 'sales', 'profit', 'ship_mode') \
    .show(10, truncate=True)

✅ Tabela clientes gravada no Delta Lake


✅ Tabela pedidos gravada no Delta Lake



Total registros inseridos: 20


+--------------+-----------+--------------------+---------------+-------+--------+------------+
|      order_id|customer_id|        product_name|       category|  sales|  profit|   ship_mode|
+--------------+-----------+--------------------+---------------+-------+--------+------------+
|CA-2016-152156|   CG-12520|Bush Somerset Col...|      Furniture| 261.96| 41.9136|Second Class|
|CA-2016-152156|   CG-12520|Hon Deluxe Fabric...|      Furniture| 731.94| 219.582|Second Class|
|CA-2016-138688|   DV-13045|Self-Adhesive Add...|Office Supplies|  14.62|  6.8714|Second Class|
|CA-2014-167164|   AG-10270|Fellowes Super St...|Office Supplies|   55.5|    9.99|Second Class|
|CA-2014-143336|   ZD-21925|          Newell 341|Office Supplies|   8.56|  2.4824|Second Class|
|US-2017-156909|   SF-20065|Global Deluxe Sta...|      Furniture| 71.372| -1.0196|Second Class|
|CA-2014-124394|   TB-21520|GBC Standard Recy...|Office Supplies|  10.78| -17.248|Second Class|
|CA-2014-124394|   TB-21520|Case Logic 2

## 4. UPDATE — Atualizando modo de envio

Pedidos com `ship_mode = 'Second Class'` serão atualizados para `'First Class'`.  
O Delta Lake marca os arquivos antigos como obsoletos e registra a operação como versão 1 (`UPDATE`).

In [4]:
from delta.tables import DeltaTable

delta_pedidos = DeltaTable.forPath(spark, f'{DELTA_PATH}/pedidos')

delta_pedidos.update(
    condition = col('ship_mode') == 'Second Class',
    set       = {'ship_mode': lit('First Class')}
)

print('✅ UPDATE — Second Class → First Class')
spark.read.format('delta').load(f'{DELTA_PATH}/pedidos') \
    .groupBy('ship_mode').count().show()

✅ UPDATE — Second Class → First Class


+--------------+-----+
|     ship_mode|count|
+--------------+-----+
|   First Class|   10|
|Standard Class|   10|
+--------------+-----+



## 5. DELETE — Removendo vendas com prejuízo

Removemos os registros onde `profit < 0` (vendas onde a loja teve prejuízo).  
O Delta Lake registra a operação como versão 2 (`DELETE`) — os dados removidos ainda são recuperáveis via Time Travel.

In [5]:
delta_pedidos.delete(condition = col('profit') < 0)

print('✅ DELETE — registros com profit < 0 removidos')
print(f'Registros restantes: {spark.read.format("delta").load(f"{DELTA_PATH}/pedidos").count()}')
spark.read.format('delta').load(f'{DELTA_PATH}/pedidos') \
    .select('order_id', 'product_name', 'category', 'sales', 'profit', 'ship_mode') \
    .show(10, truncate=True)

✅ DELETE — registros com profit < 0 removidos


Registros restantes: 14


+--------------+--------------------+---------------+-------+-------+--------------+
|      order_id|        product_name|       category|  sales| profit|     ship_mode|
+--------------+--------------------+---------------+-------+-------+--------------+
|CA-2016-152156|Bush Somerset Col...|      Furniture| 261.96|41.9136|   First Class|
|CA-2016-152156|Hon Deluxe Fabric...|      Furniture| 731.94|219.582|   First Class|
|CA-2016-138688|Self-Adhesive Add...|Office Supplies|  14.62| 6.8714|   First Class|
|CA-2014-167164|Fellowes Super St...|Office Supplies|   55.5|   9.99|   First Class|
|CA-2014-143336|          Newell 341|Office Supplies|   8.56| 2.4824|   First Class|
|US-2015-108966|Eldon Fold 'N Rol...|Office Supplies| 22.368| 2.5164|Standard Class|
|CA-2014-115812|Eldon Expressions...|      Furniture|  48.86|14.1694|Standard Class|
|CA-2014-115812|          Newell 322|Office Supplies|   7.28| 1.9656|Standard Class|
|CA-2014-115812|Mitel 5320 IP Pho...|     Technology|907.152|90.7

## 6. MERGE (UPSERT) — Atualizar ou inserir em uma operação

O `MERGE` verifica se o registro já existe:  
- Se **existe** (`order_id` encontrado) → atualiza  
- Se **não existe** → insere

In [6]:
existente = spark.read.format('delta').load(f'{DELTA_PATH}/pedidos').first()

schema_cols = ['order_id', 'customer_id', 'order_date', 'ship_date', 'ship_mode',
               'product_name', 'category', 'sub_category',
               'sales', 'quantity', 'discount', 'profit']

dados_merge = [
    # UPDATE — pedido existente ganha quantidade +1
    (existente['order_id'], existente['customer_id'],
     existente['order_date'], existente['ship_date'], existente['ship_mode'],
     existente['product_name'], existente['category'], existente['sub_category'],
     existente['sales'], existente['quantity'] + 1, existente['discount'], existente['profit']),
    # INSERT — novo pedido
    ('US-2024-NOVO01', 'CG-12520', '1/15/2024', '1/18/2024', 'First Class',
     'Apple MacBook Pro 16"', 'Technology', 'Computers', 2499.99, 1, 0.0, 499.99),
]

df_merge = spark.createDataFrame(dados_merge, schema_cols)

delta_pedidos.alias('destino').merge(
    df_merge.alias('origem'),
    'destino.order_id = origem.order_id AND destino.product_name = origem.product_name'
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print('✅ MERGE executado!')
spark.read.format('delta').load(f'{DELTA_PATH}/pedidos') \
    .select('order_id', 'product_name', 'category', 'sales', 'quantity', 'profit', 'ship_mode') \
    .orderBy('order_id') \
    .show(truncate=True)

✅ MERGE executado!


+--------------+--------------------+---------------+--------+--------+-------+--------------+
|      order_id|        product_name|       category|   sales|quantity| profit|     ship_mode|
+--------------+--------------------+---------------+--------+--------+-------+--------------+
|CA-2014-115812|Belkin F5C206VTEL...|Office Supplies|   114.9|       5|  34.47|Standard Class|
|CA-2014-115812|Chromcraft Rectan...|      Furniture|1706.184|       9|85.3092|Standard Class|
|CA-2014-115812|DXL Angle-View Bi...|Office Supplies|  18.504|       3| 5.7825|Standard Class|
|CA-2014-115812|Eldon Expressions...|      Furniture|   48.86|       7|14.1694|Standard Class|
|CA-2014-115812|Konftel 250 Confe...|     Technology| 911.424|       4|68.3568|Standard Class|
|CA-2014-115812|Mitel 5320 IP Pho...|     Technology| 907.152|       6|90.7152|Standard Class|
|CA-2014-115812|          Newell 322|Office Supplies|    7.28|       4| 1.9656|Standard Class|
|CA-2014-143336|          Newell 341|Office Suppli

## 7. Time Travel — Histórico de versões

O Delta Lake mantém um log completo de todas as operações. É possível consultar o estado dos dados em qualquer versão anterior.

In [7]:
delta_pedidos.history().select('version', 'timestamp', 'operation').show(truncate=False)

print('\n📜 Estado original (versão 0 — INSERT inicial):')
spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .load(f'{DELTA_PATH}/pedidos') \
    .select('order_id', 'product_name', 'profit', 'ship_mode') \
    .show(10, truncate=True)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|7      |2026-04-29 17:53:41.859|MERGE    |
|6      |2026-04-29 17:53:31.812|DELETE   |
|5      |2026-04-29 17:53:25.408|UPDATE   |
|4      |2026-04-29 17:53:16.188|WRITE    |
|3      |2026-04-29 17:42:19.016|MERGE    |
|2      |2026-04-29 17:41:46.352|DELETE   |
|1      |2026-04-29 17:41:19.431|UPDATE   |
|0      |2026-04-29 17:40:44.293|WRITE    |
+-------+-----------------------+---------+


📜 Estado original (versão 0 — INSERT inicial):


+--------------+--------------------+--------+------------+
|      order_id|        product_name|  profit|   ship_mode|
+--------------+--------------------+--------+------------+
|CA-2016-152156|Bush Somerset Col...| 41.9136|Second Class|
|CA-2016-152156|Hon Deluxe Fabric...| 219.582|Second Class|
|CA-2016-138688|Self-Adhesive Add...|  6.8714|Second Class|
|CA-2014-167164|Fellowes Super St...|    9.99|Second Class|
|CA-2014-143336|          Newell 341|  2.4824|Second Class|
|US-2017-156909|Global Deluxe Sta...| -1.0196|Second Class|
|CA-2014-124394|GBC Standard Recy...| -17.248|Second Class|
|CA-2014-124394|Case Logic 2.4GHz...|-17.9964|Second Class|
|US-2016-158288|Binding Machine S...|-57.7566|Second Class|
|CA-2016-128916|Staple-based wall...| -3.8208|Second Class|
+--------------+--------------------+--------+------------+
only showing top 10 rows



## 8. Estrutura de arquivos no storage

In [8]:
print('📁 Estrutura de arquivos Delta Lake:')
for root, dirs, files in os.walk(f'{DELTA_PATH}/pedidos'):
    dirs.sort()
    level  = root.replace(f'{DELTA_PATH}/pedidos', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

📁 Estrutura de arquivos Delta Lake:
pedidos/
  .part-00000-1672ca0f-ce34-4497-9388-5158d79851ac-c000.snappy.parquet.crc
  .part-00000-292c2086-8147-4b5e-ab63-ba835ab3e4ac-c000.snappy.parquet.crc
  .part-00000-4554a011-1410-4078-ac4f-95bad7e03800-c000.snappy.parquet.crc
  .part-00000-5c2c4560-4e6a-463c-a656-9e1e2e1bfce8-c000.snappy.parquet.crc
  .part-00000-c3cbac43-c222-4d24-a818-37577daaff85-c000.snappy.parquet.crc
  .part-00000-e8bf9131-4cb7-408e-a416-1110dff3c538-c000.snappy.parquet.crc
  .part-00000-eaad37ca-c3f6-4788-a6c0-a0cf6afe0e7d-c000.snappy.parquet.crc
  .part-00000-eebe5ed7-29d4-412a-ad78-eced26f892fa-c000.snappy.parquet.crc
  part-00000-1672ca0f-ce34-4497-9388-5158d79851ac-c000.snappy.parquet
  part-00000-292c2086-8147-4b5e-ab63-ba835ab3e4ac-c000.snappy.parquet
  part-00000-4554a011-1410-4078-ac4f-95bad7e03800-c000.snappy.parquet
  part-00000-5c2c4560-4e6a-463c-a656-9e1e2e1bfce8-c000.snappy.parquet
  part-00000-c3cbac43-c222-4d24-a818-37577daaff85-c000.snappy.parquet
  par